## **Imports**
---

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

USING_COLAB: bool = "google.colab" in sys.modules
DATA_ROOT: Path
WANDB_DATA_DIR: Path
GOOGLE_DRIVE_PATH: Path

def _init_colab() -> None:
    """
    Initializes the Colab environment by cloning the repository, installing dependencies, and mounting Google Drive.
    """
    REPO_URL: str = "https://github.com/Johannes-Herz/tum-adlr-13"
    REPO_BRANCH: str = "feature/shapenet-03948459-reconstruction"
    REPO_DIR: str = "/content/tum-adlr-13"

    if not os.path.exists(REPO_DIR):
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
        !pip install -q torchmetrics pydantic-settings
    else:
        !git -C {REPO_DIR} fetch origin
        !git -C {REPO_DIR} checkout {REPO_BRANCH}
        !git -C {REPO_DIR} pull

    if f"{REPO_DIR}/src" not in sys.path:
        sys.path.insert(0, f"{REPO_DIR}/src")

    !pip install torchmetrics
    !pip install pydantic-settings

    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=True)

    global GOOGLE_DRIVE_PATH
    GOOGLE_DRIVE_PATH = Path("/content/gdrive/MyDrive/tum-adlr-13")

    if not load_dotenv():
        loaded_dotenv = load_dotenv(GOOGLE_DRIVE_PATH / ".env")
        print(f"Loaded .env from Drive: {loaded_dotenv}")

    global DATA_ROOT
    DATA_ROOT = Path('/content/data')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    !cp -r /content/gdrive/MyDrive/tum-adlr-13/data/. /content/data/
    !ls /content/data/

    global WANDB_DATA_DIR
    WANDB_DATA_DIR = Path("/content/wandb")
    WANDB_DATA_DIR.mkdir(parents=True, exist_ok=True)

def _init_local() -> None:
    """
    Initializes the local environment by loading environment variables from a .env file.
    """
    %load_ext autoreload
    %autoreload 2
    load_dotenv()

    global GOOGLE_DRIVE_PATH
    GOOGLE_DRIVE_PATH = os.environ.get("GOOGLE_DRIVE_PATH")

    from tactex.settings import get_settings

    global DATA_ROOT
    DATA_ROOT = get_settings().data_root

    global WANDB_DATA_DIR
    WANDB_DATA_DIR = get_settings().project_root / "wandb"
    WANDB_DATA_DIR.mkdir(parents=True, exist_ok=True)

if USING_COLAB:
    _init_colab()
else:
    _init_local()

print("Using Colab:", USING_COLAB)
print("DATA_ROOT:", DATA_ROOT)
print("WANDB_DATA_DIR:", WANDB_DATA_DIR)
print("GOOGLE_DRIVE_PATH:", GOOGLE_DRIVE_PATH)


In [ ]:
import tactex
from tactex.reconstruction.callbacks import Checkpointer, TrainingCallback, WandBRun
from tactex.reconstruction.data import OccupancyGridDatset, build_dataloader
from tactex.reconstruction.losses import BCEDiceLoss
from tactex.reconstruction.metrics import build_metrics, build_trackers
from tactex.reconstruction.training import train
from tactex.reconstruction.unet import UNet
from tactex.settings import get_settings
from tactex.timings import get_timings, set_enabled

print(tactex.__file__)

In [ ]:
from collections.abc import Callable
from pathlib import Path

import torch
from IPython.display import clear_output
from torch import nn, optim
from torch.utils.data import DataLoader
from torchmetrics import MetricCollection


In [ ]:
import wandb

wandb.login()

## **Config**
---

In [ ]:
RESOLUTION: int = 64

DEVICE: torch.device = torch.accelerator.current_accelerator(check_available=True)
if DEVICE is None:
    DEVICE = torch.device("cpu")

print("Resolution:", RESOLUTION)
print("Device:", DEVICE)

In [ ]:
WANDB_ENTITY: str = "tum-adlr-13"
WANDB_PROJECT: str = "03948459"

## **Dataset**
---

In [ ]:
dataset_directory: Path = DATA_ROOT / "datasets" / "03948459" / f"res{str(RESOLUTION).zfill(3)}" / "v001"
train_dataset: OccupancyGridDatset = OccupancyGridDatset(root=dataset_directory, split="train", in_memory=True)
val_dataset: OccupancyGridDatset = OccupancyGridDatset(root=dataset_directory, split="val", in_memory=True)

print("Train Dataset Length: ", len(train_dataset))
print("Val Dataset Length: ", len(val_dataset))

In [ ]:
BATCH_SIZE: int = 64
NUM_WORKERS: int = 2

train_dataloader: DataLoader = build_dataloader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, resolution=RESOLUTION, max_rays=50, seed=None, num_workers=NUM_WORKERS, name="train.dataloader")
val_dataloader: DataLoader = build_dataloader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, resolution=RESOLUTION, max_rays=50, seed=100, num_workers=NUM_WORKERS, name="val.dataloader")

## **Training**
---

In [ ]:
depth: int = 5
width: int = 32
kernel_size: int = 3
learning_rate: float = 3e-4

config = {
    "depth": depth,
    "width": width,
    "kernel_size": kernel_size,
    "learning_rate": learning_rate,
}

In [ ]:
unet: UNet = UNet.from_depth(in_channels=3, out_channels=1, depth=config["depth"], width=config["width"], kernel_size=config["kernel_size"])
loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor] = BCEDiceLoss() # nn.BCEWithLogitsLoss()
optimizer: optim.Adam = optim.Adam(params=unet.parameters(), lr=learning_rate)
metrics: MetricCollection = build_metrics(loss_fn=loss_fn, threshold=0.5)
train_tracker, val_tracker = build_trackers(metrics=metrics, device=DEVICE)
lr_scheduler: optim.lr_scheduler.ReduceLROnPlateau =  optim.lr_scheduler.ReduceLROnPlateau(
    optimizer=optimizer, 
    mode="min",
    factor=0.5, 
    patience=4,
    min_lr=1e-6
)

In [ ]:
WANDB_RUN_ID: str = wandb.sdk.lib.runid.generate_id()
print(f"WandB Run ID: {WANDB_RUN_ID}")
WANDB_RUN_NAME: str = f"unet-pistol-d{config['depth']}-w{config['width']}-k{config['kernel_size']}-lr{config['learning_rate']:.0e}-loss-bce-dice"
print(f"WandB Run Name: {WANDB_RUN_NAME}")

In [ ]:
def run_training():
    EPOCHS: int = 50
    with WandBRun(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        id=WANDB_RUN_ID,
        name=WANDB_RUN_NAME,
        dir=WANDB_DATA_DIR,
        config=config,
    ) as wandb_run:
        get_timings().reset()
        val_batches = list(val_dataloader)
        checkpoint_callback: Checkpointer = Checkpointer(
            save_dir=DATA_ROOT / "models" / WANDB_PROJECT / WANDB_RUN_ID,
            every_n_epochs=5,
            metric="val/iou_occ",
            mode="max",
            log_to_wandb=True
        )
        callbacks: list[TrainingCallback] = [checkpoint_callback, wandb_run]
        train(model=unet, train_dataloader=train_dataloader, val_dataloader=val_batches, loss_fn=loss_fn, optimizer=optimizer, lr_scheduler=lr_scheduler, train_tracker=train_tracker, val_tracker=val_tracker, epochs=EPOCHS, device=DEVICE, callbacks=callbacks)


In [ ]:
run_training()

## **Sweeping**
---

In [ ]:
def run_sweep():
    with WandBRun() as wandb_run:
        clear_output(wait=True)
        set_enabled(False)

        config: dict = wandb_run.run.config
        unet: UNet = UNet.from_depth(in_channels=3, out_channels=1, depth=config["depth"], width=config["width"], kernel_size=config["kernel_size"])
        optimizer: optim.Adam = optim.Adam(params=unet.parameters(), lr=config["learning_rate"])
        loss_fn: Callable[[torch.Tensor, torch.Tensor], torch.Tensor] = nn.BCEWithLogitsLoss()
        metrics: MetricCollection = build_metrics(loss_fn=loss_fn, threshold=0.5)
        train_tracker, val_tracker = build_trackers(metrics=metrics, device=DEVICE)
        lr_scheduler: optim.lr_scheduler.LRScheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer=optimizer, 
            mode="min",
            factor=0.5, 
            patience=4,
            min_lr=1e-6
        )
        val_batches = list(val_dataloader)
        
        checkpoint_callback: Checkpointer = Checkpointer(
            save_dir=DATA_ROOT / "models" / wandb_run.run.project / wandb_run.run.id,
            every_n_epochs=5,
            metric="val/iou_occ", 
            mode="max",
            log_to_wandb=True
        )
        callbacks: list[TrainingCallback] = [checkpoint_callback, wandb_run]
        train(
            model=unet,
            loss_fn=loss_fn,
            optimizer=optimizer,
            train_dataloader=train_dataloader,
            val_dataloader=val_batches,
            train_tracker=train_tracker,
            val_tracker=val_tracker,
            epochs=20,
            device=DEVICE,
            lr_scheduler=lr_scheduler,
            callbacks=callbacks
        )

In [ ]:
sweep_config: dict = {
    "name": "unet-pistol-sweep",
    "method": "random",
    "metric": {
        "name": "val/iou_occ",
        "goal": "maximize"
    },
    "early_terminate": {
        "type": "hyperband",
        "min_iter": 5,
    },
    "parameters": {
        "width": {
            "values": [16, 32, 64]
        },
        "depth": {
            "values": [2, 3, 4, 5]
        },
        "kernel_size": {
            "values": [3, 5]
        },
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 1e-5,
            "max": 1e-2
        }
    }
}

In [ ]:
sweep_id: str = wandb.sweep(sweep_config, project=WANDB_PROJECT, entity=WANDB_ENTITY)
wandb.agent(sweep_id, function=run_sweep, count=20, project=WANDB_PROJECT, entity=WANDB_ENTITY)

## **Google Drive**
---

In [ ]:
def move_models_to_drive(wandb_run_id: str = WANDB_RUN_ID) -> None:
    """
    Moves the trained models from the local data directory to Google Drive.
    """
    import shutil

    source_dir: Path = DATA_ROOT / "models" / "03948459" / wandb_run_id
    destination_dir: Path = GOOGLE_DRIVE_PATH / "models" / "03948459" / wandb_run_id
    if not destination_dir.exists():
        destination_dir.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)

move_models_to_drive()